In [2]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import open3d as o3d
from scipy.spatial.transform import Rotation as R 
import os 
import glob 
import pandas as pd 

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


Thông số camera

In [3]:
color_intrinsics = {
    'width': 1280,
    'height': 720,
    'fx': 643.90087890625,
    'fy': 643.1365356445312,
    'cx': 650.2113037109375,
    'cy': 355.79559326171875,
    'model': "distortion.inverse_brown_conrady",
    'coeffs': [-0.05658450722694397, 0.06544225662946701,-0.0008694113348610699, 0.00016751799557823688,-0.020957745611667633]
}

depth_intrinsics = {
    'width': 1280,
    'height': 720,
    'fx': 650.0616455078125,
    'fy': 650.0616455078125,
    'cx': 649.5928955078125,
    'cy': 360.9415588378906,
    'model': "distortion.brown_conrady",
    'coeffs': [0.0, 0.0, 0.0, 0.0, 0.0]
}

R_depth_to_color = np.array([

    [0.9999898076057434,    -0.00020347206736914814, -0.004507721401751041],
    [0.00018898719281423837, 0.9999948143959045,     -0.0032135415822267532],
    [0.004508351907134056,   0.003212657058611512,    0.9999846816062927]

    ])
t_depth_to_color = np.array([

    [-0.05905],
    [8.67399e-5],
    [0.00041]
    
    ])

In [4]:
def get_yolo_results_for_image(image_name, yolo_txt_dir):
    """
    Đọc file .txt kết quả YOLO tùy chỉnh và trích xuất bbox.
    Định dạng file dự kiến: {classname} {conf} {xmin} {xmax} {ymin} {ymax} ...
    
    Args:
        image_name (str): Tên file ảnh (ví dụ: "0000.png")
        yolo_txt_dir (str): Đường dẫn tới thư mục chứa file .txt
        
    Returns:
        list: Danh sách các bbox [[x_min, y_min, x_max, y_max], ...]
    """
    
    # 1. Tạo đường dẫn file .txt từ tên file ảnh
    base_name = os.path.splitext(image_name)[0]
    txt_name = base_name + ".txt"
    txt_path = os.path.join(yolo_txt_dir, txt_name)
    
    bboxes = []
    
    # 2. Kiểm tra file .txt có tồn tại không
    if not os.path.exists(txt_path):
        return []

    # 3. Đọc và xử lý file
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # 4. Parse 
                # {classname} {conf} {xmin} {xmax} {ymin} {ymax} ...
                parts = line.split()
                
                # Cần ít nhất 6 phần tử (class, conf, xmin, xmax, ymin, ymax)
                if len(parts) < 6:
                    print(f"  -> Cảnh báo: Dòng không hợp lệ trong {txt_name}: {line}")
                    continue
                    
                # 5. Trích xuất tọa độ pixel trực tiếp
                try:
                    x_min = int(float(parts[2]))
                    x_max = int(float(parts[3]))
                    y_min = int(float(parts[4]))
                    y_max = int(float(parts[5]))
                
                    # 6. Thêm vào list
                    bboxes.append([x_min, y_min, x_max, y_max])
                    
                except ValueError:
                    print(f"  -> Cảnh báo: Không thể parse tọa độ trong {txt_name}: {line}")

    except Exception as e:
        print(f"  -> Lỗi khi đọc file YOLO {txt_path}: {e}")
        return []
        
    return bboxes

In [5]:
def map_depth_to_color_3d(depth_img, color_img, depth_intr, color_intr, R_depth_to_color, t_depth_to_color):
    """
    Chuyển toàn bộ ảnh depth sang point cloud 3D trong hệ tọa độ color, với xử lý biến dạng và nội suy màu.
    
    Args:
        depth_img: (H, W) ảnh depth (mm)
        color_img: (Hc, Wc, 3) ảnh color
        depth_intr: dict {'fx','fy','cx','cy'}
        color_intr: dict {'fx','fy','cx','cy', 'coeffs'}
        R_depth_to_color: (3,3) ma trận xoay từ depth -> color
        t_depth_to_color: (3,) vector tịnh tiến từ depth -> color
    
    Returns:
        points_color_valid: (N,3) toạ độ 3D trong hệ color
        colors_valid: (N,3) màu RGB tương ứng [0-1]
    """
    # Chuẩn bị lưới pixel depth (giữ float32 để độ chính xác subpixel)
    h, w = depth_img.shape
    u, v = np.meshgrid(np.arange(w, dtype=np.float32),
                      np.arange(h, dtype=np.float32))
    
    # Chuyển depth sang mét và lọc giá trị không hợp lệ
    Z = depth_img.astype(np.float32) / 1000.0
    valid = (Z > 0.01) & (Z < 5.0) & np.isfinite(Z)
    
    # Tùy chọn: điền lỗ nhỏ bằng bộ lọc trung vị
  # bật lên nếu cần
    # from scipy.ndimage import median_filter
    # Z_filled = median_filter(Z, size=3)
    # Z[~valid] = Z_filled[~valid]
    # valid = (Z > 0.01) & (Z < 5.0) & np.isfinite(Z)
    
    # Back-project sang 3D trong hệ depth (vector hóa)
    X = (u - depth_intr['cx']) * Z / depth_intr['fx']
    Y = (v - depth_intr['cy']) * Z / depth_intr['fy']
    points_depth = np.stack((X, Y, Z), axis=-1).reshape(-1, 3)
    valid_flat = valid.reshape(-1)
    points_depth_valid = points_depth[valid_flat]
    
    if len(points_depth_valid) == 0:
        return np.array([]), np.array([])
    
    # Biến đổi sang hệ color: p_c = R @ p_d + t
    points_color = (R_depth_to_color @ points_depth_valid.T).T + t_depth_to_color.reshape(1, 3)
    
    # Chiếu sang ảnh color (giữ tọa độ float để lấy mẫu subpixel)
    Xc, Yc, Zc = points_color[:, 0], points_color[:, 1], points_color[:, 2]
    valid_z = Zc > 1e-6
    points_color_valid = points_color[valid_z]
    
    # Chiếu với hiệu chỉnh biến dạng nếu camera màu có hệ số biến dạng
    if 'coeffs' in color_intr and np.any(np.abs(color_intr['coeffs']) > 1e-8):
        # Tọa độ chuẩn hóa
        xn = Xc[valid_z] / Zc[valid_z]
        yn = Yc[valid_z] / Zc[valid_z]
        # Áp dụng biến dạng Brown-Conrady
        r2 = xn*xn + yn*yn
        k1,k2,p1,p2,k3 = color_intr['coeffs']
        radial = 1 + k1*r2 + k2*r2*r2 + k3*r2*r2*r2
        xd = xn*radial + 2*p1*xn*yn + p2*(r2 + 2*xn*xn)
        yd = yn*radial + p1*(r2 + 2*yn*yn) + 2*p2*xn*yn
        # Chiếu sang pixel
        u_c = xd * color_intr['fx'] + color_intr['cx']
        v_c = yd * color_intr['fy'] + color_intr['cy']
    else:
        # Không có biến dạng - chiếu trực tiếp
        u_c = Xc[valid_z] * color_intr['fx'] / Zc[valid_z] + color_intr['cx']
        v_c = Yc[valid_z] * color_intr['fy'] / Zc[valid_z] + color_intr['cy']
    
    #Lấy mẫu màu dùng nội suy song tuyến
    h_c, w_c = color_img.shape[:2]
    in_bounds = (u_c >= 0) & (u_c <= w_c-1) & (v_c >= 0) & (v_c <= h_c-1)
    
    points_color_valid = points_color_valid[in_bounds]
    u_valid = u_c[in_bounds]
    v_valid = v_c[in_bounds]
    
    # Nội suy song tuyến
    u0 = np.floor(u_valid).astype(np.int32)
    v0 = np.floor(v_valid).astype(np.int32)
    u1 = np.minimum(u0 + 1, w_c-1)
    v1 = np.minimum(v0 + 1, h_c-1)
    
    wu1 = (u_valid - u0).reshape(-1,1)
    wu0 = (1 - wu1)
    wv1 = (v_valid - v0).reshape(-1,1)
    wv0 = (1 - wv1)
    
    c00 = color_img[v0, u0].astype(np.float32)
    c01 = color_img[v0, u1].astype(np.float32)
    c10 = color_img[v1, u0].astype(np.float32)
    c11 = color_img[v1, u1].astype(np.float32)
    
    colors_valid = (wu0*wv0*c00 + wu1*wv0*c01 + wu0*wv1*c10 + wu1*wv1*c11)
    colors_valid = colors_valid[...,::-1] / 255.0  # BGR→RGB
    
    return points_color_valid, colors_valid

In [6]:
def get_point_cloud_in_box(points_3d, colors, box_2d):
    """
    Lọc điểm trong bounding box 2D
    Args:
        points_3d: (N,3) điểm trong hệ color
        colors: (N,3) màu RGB [0-1]
        box_2d: (x_min, y_min, x_max, y_max) trong ảnh màu
    Returns:
        points_in_box: (M,3) điểm trong box
        colors_in_box: (M,3) màu tương ứng
    """
    x_min, y_min, x_max, y_max = box_2d
    
    # Chiếu ngược điểm 3D về ảnh màu
    
    # Chiếu điểm về pixel
    X, Y, Z = points_3d[:, 0], points_3d[:, 1], points_3d[:, 2]
    u = X * color_intrinsics['fx'] / Z + color_intrinsics['cx'] 
    v = Y * color_intrinsics['fy'] / Z + color_intrinsics['cy']
    
    # Lọc điểm trong box
    in_box = (u >= x_min) & (u < x_max) & (v >= y_min) & (v < y_max)
    points_in_box = points_3d[in_box]
    colors_in_box = colors[in_box]
    
    return points_in_box, colors_in_box

In [7]:
# Axis-balanced and plane-fit centroid methods
def axis_balanced_centroid(points, use_median_z=True, eps=1e-8):
    """
    Compute a centroid that balances the influence of X/Y/Z by whitening each axis
    (divide by per-axis std) before computing centroid, then un-whiten.

    Args:
        points: (N,3) ndarray of points in color frame
        use_median_z: if True, replace returned Z with the median Z (robust)
        eps: small value to avoid division by zero

    Returns:
        centroid: (3,) ndarray
    """
    if points.shape[0] == 0:
        return np.array([np.nan, np.nan, np.nan])

    # Compute per-axis std (use robust estimator if desired)
    std = np.std(points, axis=0)
    std_safe = np.where(std < eps, 1.0, std)

    # Whiten axes: scale each axis to unit variance
    scale = 1.0 / std_safe
    whitened = points * scale  # broadcasting

    # Centroid in whitened space
    centroid_whiten = np.mean(whitened, axis=0)

    # Un-whiten
    centroid = centroid_whiten / scale

    # Optionally force Z to robust measure (median) because Z is often noisier
    if use_median_z:
        centroid[2] = np.median(points[:, 2])

    return centroid


In [8]:
# 1. TẠO DANH SÁCH ẢNH CẦN XỬ LÝ
base_path = r"D:\HaAnh\AI_race\train"
rgb_files = sorted(glob.glob(os.path.join(base_path, "rgb", "*.png")))
depth_files = sorted(glob.glob(os.path.join(base_path, "depth", "*.png")))

YOLO_TXT_DIR = r"D:\HaAnh\AI_race\yolo_result"

# 2. LIST ĐỂ LƯU KẾT QUẢ CUỐI CÙNG
all_final_outputs = []

# 3. VÒNG LẶP BÊN NGOÀI
for rgb_path, depth_path in zip(rgb_files, depth_files):
    
    IMAGE_FILENAME = os.path.basename(rgb_path)
    print(f"\n=================================================")
    print(f"ĐANG XỬ LÝ ẢNH: {IMAGE_FILENAME}")
    print(f"=================================================")

    depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    img = cv2.imread(rgb_path, cv2.IMREAD_UNCHANGED)
    
    if depth is None or img is None:
        print(f"  -> Lỗi: Không load được ảnh {IMAGE_FILENAME}")
        continue
        
    all_yolo_bboxes = get_yolo_results_for_image(
        IMAGE_FILENAME, 
        YOLO_TXT_DIR
    )
    
    if not all_yolo_bboxes:
        print("  -> Không tìm thấy bounding box nào.")
        all_final_outputs.append((IMAGE_FILENAME, None, None, None))
        continue
        
    print(f"Bắt đầu Trích xuất 3D points cho {len(all_yolo_bboxes)} ứng cử viên...")

    candidate_parcels_info = []
    for i, bbox in enumerate(all_yolo_bboxes):
        try:
            # points_3d, colors_3d = get_point_cloud_from_bbox(
            #     bbox, depth, img, 
            #     depth_intrinsics, color_intrinsics, 
            #     R_depth_to_color, t_depth_to_color
            # )

            points_full, colors_full = map_depth_to_color_3d(depth, img, depth_intrinsics, color_intrinsics,
                                                R_depth_to_color, t_depth_to_color)
            
            points_3d, colors_3d = get_point_cloud_in_box(points_full, colors_full, bbox)

            if len(points_3d) > 0:
                candidate_parcels_info.append({
                    'points_3d': points_3d,
                    'colors_3d': colors_3d,
                    'original_bbox': bbox
                })
        except Exception as e:
            print(f"  -> Lỗi khi xử lý bbox {bbox}: {e}")
            
    # --- BƯỚC 4: TÍNH TÂM 3D ---
    print("\nBắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số)...")
    final_candidates = []

    for i, parcel_info in enumerate(candidate_parcels_info):
        parcel_points_3d = parcel_info['points_3d']
        if len(parcel_points_3d) < 100:
            print(f"  -> Bỏ qua ứng cử viên {i}: quá ít điểm.")
            continue

        #--- Lọc nhiễu không gian ---
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(parcel_points_3d)
        pcd_clean, ind = pcd.remove_statistical_outlier(nb_neighbors=30, std_ratio=1.0)
        filtered_points = np.asarray(pcd_clean.points)

        # --- Trung bình có trọng số ---
        mean = np.mean(filtered_points, axis=0)
        dist = np.linalg.norm(filtered_points - mean, axis=1)
        mask = dist < np.percentile(dist, 80)
        stable_points = filtered_points[mask]

        # center_3d = np.median(parcel_points_3d, axis=0)
        center_3d = axis_balanced_centroid(stable_points)

        final_candidates.append({
            'center_3d': center_3d,
            'original_info': parcel_info
        })

        print(f"  Ứng cử viên {i}: Tâm 3D (sau lọc) = ({center_3d[0]:.3f}, {center_3d[1]:.3f}, {center_3d[2]:.3f})")

    # --- BƯỚC 5: CHỌN BƯU KIỆN CUỐI CÙNG ---
    print("\nBắt đầu Chọn bưu kiện cuối cùng...")
    top_parcel = None
    num_candidates = len(final_candidates)
    
    if num_candidates == 0:
        print("[KẾT QUẢ]: Không có ứng cử viên hợp lệ.")
        all_final_outputs.append((IMAGE_FILENAME, None, None, None)) # Chỉ 3 cột x,y,z
        continue # Thêm continue ở đây
        
    elif num_candidates == 1:
        top_parcel = final_candidates[0]
        print(f"[KẾT QUẢ]: Chọn ứng cử viên duy nhất.")
    else:
        print(f"Phát hiện {num_candidates} ứng cử viên. Áp dụng luật Tie-Break...")
        
        # ==================================================================
        # === SỬA LOGIC: "TRÊN CÙNG" = Z NHỎ NHẤT ===
        # ==================================================================
        
        # Sắp xếp theo Z TĂNG DẦN (reverse=False) để tìm vật GẦN NHẤT
        final_candidates.sort(key=lambda c: c['center_3d'][2], reverse=False) 
        
        # Đây là Z của vật gần nhất
        z_min = final_candidates[0]['center_3d'][2] 
        
        # Nhóm các bưu kiện có chênh lệch độ cao < 5mm
        # So sánh với z_min, không phải z_max
        tie_group = [c for c in final_candidates if abs(c['center_3d'][2] - z_min) < 0.005]
        print(f"  -> {len(tie_group)} bưu kiện có cùng độ cao GẦN NHẤT (±5mm).")
        # ==================================================================
        
        if len(tie_group) == 1:
            top_parcel = tie_group[0]
        else:
            # Ưu tiên bưu kiện xa robot nhất (|Y| lớn nhất)
            # Ràng buộc này trong đề bài là đúng, code bạn đã làm đúng
            tie_group.sort(key=lambda c: abs(c['center_3d'][1]), reverse=True)
            top_parcel = tie_group[0]
            print(f"  -> Chọn vật xa robot nhất (|Y| lớn nhất).")

    if top_parcel is not None:
        c = top_parcel['center_3d']
        print(f"\n--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH {IMAGE_FILENAME}] ---")
        print(f"  Tâm gắp (x, y, z): ({c[0]:.4f}, {c[1]:.4f}, {c[2]:.4f})")
        # Chỉ lưu 3 cột x,y,z
        all_final_outputs.append((IMAGE_FILENAME, c[0], c[1], c[2]))
    else:
        print(f"\n[KẾT QUẢ]: Không chọn được bưu kiện nào.")
        # Chỉ lưu 3 cột x,y,z
        all_final_outputs.append((IMAGE_FILENAME, None, None, None))

print("\n=== HOÀN TẤT QUÁ TRÌNH ===")


ĐANG XỬ LÝ ẢNH: 0000.png
Bắt đầu Trích xuất 3D points cho 1 ứng cử viên...

Bắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số)...
  Ứng cử viên 0: Tâm 3D (sau lọc) = (-0.078, 0.040, 1.074)

Bắt đầu Chọn bưu kiện cuối cùng...
[KẾT QUẢ]: Chọn ứng cử viên duy nhất.

--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH 0000.png] ---
  Tâm gắp (x, y, z): (-0.0779, 0.0399, 1.0738)

ĐANG XỬ LÝ ẢNH: 0001.png
Bắt đầu Trích xuất 3D points cho 2 ứng cử viên...

Bắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số)...
  Ứng cử viên 0: Tâm 3D (sau lọc) = (-0.074, -0.166, 1.179)
  Ứng cử viên 1: Tâm 3D (sau lọc) = (-0.079, 0.041, 1.073)

Bắt đầu Chọn bưu kiện cuối cùng...
Phát hiện 2 ứng cử viên. Áp dụng luật Tie-Break...
  -> 1 bưu kiện có cùng độ cao GẦN NHẤT (±5mm).

--- [KẾT QUẢ CUỐI CÙNG CHO ẢNH 0001.png] ---
  Tâm gắp (x, y, z): (-0.0788, 0.0412, 1.0731)

ĐANG XỬ LÝ ẢNH: 0002.png
Bắt đầu Trích xuất 3D points cho 2 ứng cử viên...

Bắt đầu Tính toán Tâm 3D (lọc outlier + trung bình có trọng số

In [9]:
df = pd.DataFrame(all_final_outputs, 
                  columns=['image_filename', 'x', 'y', 'z'])

output_csv_path = "submission4.csv"
df.to_csv(output_csv_path, index=False, float_format='%.6f')

print(f"Đã lưu kết quả ra file: {output_csv_path}")
df.head()


Đã lưu kết quả ra file: submission4.csv


,image_filename,x,y,z
0,0000.png,-0.077895,0.039855,1.073780
1,0001.png,-0.078756,0.041206,1.073145
2,0002.png,-0.129347,-0.148541,1.055401
3,0003.png,-0.127198,-0.183881,1.003519
4,0004.png,0.083718,-0.203959,1.026456


In [10]:
gt = pd.read_csv(r"D:\HaAnh\AI_race\public_train.csv")
pred = pd.read_csv(r"D:\HaAnh\AI_race\submission4.csv")

# chuẩn hoá tên trước khi merge:
pred['image_filename'] = pred['image_filename'].apply(
    lambda x: x if x.startswith('image_') else f"image_{x}"
)

merged = pd.merge(gt, pred, on='image_filename', suffixes=('_gt', '_pred'))

merged['err'] = np.sqrt(
    (merged['x_gt'] - merged['x_pred'])**2 +
    (merged['y_gt'] - merged['y_pred'])**2 +
    (merged['z_gt'] - merged['z_pred'])**2
)

merged['MCE_i'] = np.minimum(merged['err'] / 0.05, 1.0)

MCE = merged['MCE_i'].mean()

print(f"Mean Center Error (MCE): {MCE:.4f}")
print(f"→ Điểm tương ứng: {(1 - MCE)*100:.2f}%")

print("\nThống kê thêm:")
print(f"  Trung vị sai số: {merged['err'].median()*1000:.2f} mm")
print(f"  Trung bình sai số: {merged['err'].mean()*1000:.2f} mm")
print(f"  % ảnh hợp lệ (err ≤ 5cm): {(merged['err'] <= 0.05).mean()*100:.2f}%")


Mean Center Error (MCE): 0.6754
→ Điểm tương ứng: 32.46%

Thống kê thêm:
  Trung vị sai số: 30.63 mm
  Trung bình sai số: 59.99 mm
  % ảnh hợp lệ (err ≤ 5cm): 70.00%
